# 3 - CHUNK AND EMBED THE CORPUS

A lead-only baseline - title plus the first 400 words of each page - reached **R@1 64.0 / R@10 93.6 /
R@20 96.4** (bge + prefix), beating BM25 on the same input by 11.3 points of R@1.

This notebook asks whether using the WHOLE page helps. Chunk every page, embed every
chunk once, then reduce chunks to a page score two ways:

| variant | page score | vectors kept |
|---|---|---|
| **mean** | mean of the page's chunk vectors, re-normalised | 1 per page |
| **max** | the single best-matching chunk | all chunks |

Both are reductions of the SAME chunk matrix, so the second costs seconds once the first
is paid for. Prediction on record before running: **max > lead-only > mean**. Mean pools a
many-subtopic article into a centroid that matches nothing sharply; max dilutes nothing.
(That ordering is a prior from memory, NOT a checked citation - this measures it.)

Three configurations are being compared, which is selection. At n=16,229 the standard
error on a ~96% recall is ~0.15 points, so a gap above ~0.5 points is real. The write-up
reports **all three rows**, never just the winner - that is what keeps this honest without
a held-out split.

In [ ]:
# §0 SETUP
import os, sys, json, pickle, time
sys.path.insert(0, os.getcwd())
import numpy as np
import nqrun
import torch as _t
from sentence_transformers import SentenceTransformer

In [ ]:
# §1 CONFIG - identical model and query prefix to the lead-only baseline, so the only change is
# how a page's text becomes a page score. Anything else would confound the comparison.
MODEL   = 'BAAI/bge-base-en-v1.5'
CHUNK   = 200          # words per chunk
OVERLAP = 50           # words shared with the previous chunk -> stride 150
KS      = (1, 5, 10, 20)
PROBE   = 2000

BATCH = 256 if _t.cuda.is_available() else 32

# The query prefix was MEASURED as worth +1.5 R@1 / +0.014 MRR on this corpus. Not re-litigated.
QPREFIX = 'Represent this sentence for searching relevant passages: '

# The lead-only numbers, hardcoded so the final table is self-contained.
LEAD_ONLY = {'recall': {1: 0.640, 5: 0.886, 10: 0.936, 20: 0.964}, 'mrr': 0.748}

N_PAGES_EXPECTED, N_QS_EXPECTED = 18997, 16229
OUT, HF_DIR = nqrun.art('step2-retrieval'), 'step2-retrieval'
print(f'{MODEL} | chunk {CHUNK}/{OVERLAP} | batch {BATCH}')
print(f'root {nqrun.ROOT} | run {nqrun.RUN_ID} | {nqrun.gpu_info()}')

## Definitions

Nothing below executes; running these cells only defines names.

In [ ]:
# DEFINITIONS — the metric reporter every row goes through, so the three
# configurations are measured by identical code.
def report(name, ranks):
    """Print recall@k and MRR for one ranking, and record it in `res`."""
    rec = {k: float((ranks < k).mean()) for k in KS}
    mrr = float((1.0 / (ranks + 1)).mean())
    print(f'{name:<22} ' + '  '.join(f'R@{k} {rec[k]:6.1%}' for k in KS) +
          f'  MRR {mrr:.3f}  median rank {int(np.median(ranks))}')
    res[name] = {'recall': rec, 'mrr': mrr, 'median_rank': int(np.median(ranks))}
    return ranks

## Orchestration

In [ ]:
# §2 LOAD + GUARD. From HF always: the local step1-clean pickle is a shard-0 file left by a verification run.
C = pickle.load(open(nqrun.hf_pull('step1-clean/clean_corpus.pkl'), 'rb'))
docs, questions = C['documents'], C['questions']
assert len(docs) == N_PAGES_EXPECTED, f'{len(docs)} pages != {N_PAGES_EXPECTED} - WRONG ARTIFACT'
assert len(questions) == N_QS_EXPECTED, f'{len(questions)} questions != {N_QS_EXPECTED} - WRONG ARTIFACT'
print(f'{len(docs):,} pages | {len(questions):,} questions | guards OK')

In [ ]:
# §3 CHUNK. Title prepended to every chunk: a chunk from the middle of an article has no
# other way to say what it is about, and the question almost always names the entity.
# This matches the lead-only baseline, whose lead text also carried the title.
H = list(docs)
stride = CHUNK - OVERLAP
texts, owner = [], []
for pi, h in enumerate(H):
    w = docs[h]['words']
    title = docs[h]['title']
    for a in range(0, max(len(w), 1), stride):
        part = w[a:a + CHUNK]
        if not part:
            break
        texts.append(f"{title}. {' '.join(part)}")
        owner.append(pi)                       # page ROW index, not the hash
owner = np.array(owner, dtype='int32')
print(f'{len(texts):,} chunks from {len(H):,} pages ({len(texts) / len(H):.1f} per page)')

# GATES that can fail.
print(f'  pages with no chunk : {len(H) - len(np.unique(owner))}        <- must be 0')
cpp = np.bincount(owner, minlength=len(H))
print(f'  chunks per page     : min {cpp.min()} | median {int(np.median(cpp))} | max {cpp.max()}')
print(f'  index if max kept   : {len(texts) * 768 * 4 / 1e9:.2f} GB float32')

In [ ]:
# §4 EMBED every chunk ONCE. Both variants are reductions of this matrix.

dev = 'cuda' if _t.cuda.is_available() else 'cpu'
m = SentenceTransformer(MODEL, device=dev)
m.max_seq_length = 512
ntok = np.array([len(x) for x in m.tokenizer(texts[:2000])['input_ids']])
print(f'chunk tokens: median {int(np.median(ntok))} | p95 {int(np.percentile(ntok, 95))} '
      f'| over 512: {(ntok > 512).mean():.1%}')

t0 = time.time()
m.encode(texts[:PROBE], batch_size=BATCH, normalize_embeddings=True, show_progress_bar=False)
rate = PROBE / (time.time() - t0)
print(f'throughput {rate:,.0f} chunks/s -> {len(texts) / rate / 60:.1f} min for all '
      f'{len(texts):,} chunks')

t0 = time.time()
E = np.asarray(m.encode(texts, batch_size=BATCH, normalize_embeddings=True,
                        show_progress_bar=True), dtype='float32')
print(f'chunks {E.shape} in {time.time() - t0:.0f}s')

In [ ]:
# §5 REDUCE + SCORE. Rank of the gold page = how many pages score strictly higher. Exact,
# no top-N cutoff, identical metric code to the baseline so the rows are comparable.
row = {h: i for i, h in enumerate(H)}
gi = np.array([row[q['doc_hash']] for q in questions])
qtext = [q['question'] for q in questions]
res = {}

Q = np.asarray(m.encode([QPREFIX + t for t in qtext], batch_size=BATCH,
                        normalize_embeddings=True, show_progress_bar=True), dtype='float32')

# --- mean: sum chunk vectors per page, then RE-NORMALISE. Without the renormalise, a
#     page with many chunks gets a smaller norm and cosine is silently distorted.
P = np.zeros((len(H), E.shape[1]), dtype='float32')
np.add.at(P, owner, E)
P /= np.maximum(np.linalg.norm(P, axis=1, keepdims=True), 1e-12)

r_mean = np.empty(len(Q), dtype='int32')
for a in range(0, len(Q), 2048):
    S = Q[a:a + 2048] @ P.T
    g = S[np.arange(len(S)), gi[a:a + 2048]][:, None]
    r_mean[a:a + 2048] = (S > g).sum(1)
report('chunk mean', r_mean)

# --- max: best chunk per page. Scatter-max over the chunk scores, in query blocks so the
#     score matrix stays small (16,229 x ~460k would be 30 GB).
Et = _t.from_numpy(E).to(dev)
own_t = _t.from_numpy(owner.astype('int64')).to(dev)
r_max = np.empty(len(Q), dtype='int32')
BLK = 256
for a in range(0, len(Q), BLK):
    q = _t.from_numpy(Q[a:a + BLK]).to(dev)
    S = q @ Et.T                                               # (b, n_chunks)
    page = _t.full((S.shape[0], len(H)), -2.0, device=dev)
    page.scatter_reduce_(1, own_t.expand(S.shape[0], -1), S, reduce='amax')
    g = page.gather(1, _t.from_numpy(gi[a:a + BLK].astype('int64')).to(dev)[:, None])
    r_max[a:a + BLK] = (page > g).sum(1).cpu().numpy()
    del S, page
report('chunk max', r_max)

print(f"\n{'lead-only baseline':<22} " +
      '  '.join(f"R@{k} {LEAD_ONLY['recall'][k]:6.1%}" for k in KS) +
      f"  MRR {LEAD_ONLY['mrr']:.3f}")
for k in KS:
    d_mean = res['chunk mean']['recall'][k] - LEAD_ONLY['recall'][k]
    d_max = res['chunk max']['recall'][k] - LEAD_ONLY['recall'][k]
    print(f'  R@{k:<3} vs lead-only:  mean {d_mean:+.1%}   max {d_max:+.1%}')

In [ ]:
# §6 WHERE DOES CHUNKING PAY? The headline deltas hide the mechanism. The claim behind
# chunking is that it rescues questions whose answer is NOT in the lead; this tests it.
LEAD = 400
in_lead = np.array([q['gold'][0][2] <= LEAD for q in questions])
plen = np.array([len(docs[q['doc_hash']]['words']) for q in questions])

print(f'answer inside the first {LEAD} words: {in_lead.mean():.1%} of questions\n')
print(f"{'subset':<28}{'n':>8}{'mean R@20':>12}{'max R@20':>12}")
for label, sel in [('answer in lead', in_lead), ('answer NOT in lead', ~in_lead),
                   ('short page (<1k words)', plen < 1000),
                   ('long page (>=4k words)', plen >= 4000)]:
    print(f'{label:<28}{sel.sum():>8,}{(r_mean[sel] < 20).mean():>12.1%}'
          f'{(r_max[sel] < 20).mean():>12.1%}')

# questions max rescues that mean loses, and vice versa - the concrete trade
rescued = np.where((r_max < 20) & (r_mean >= 20))[0]
lost = np.where((r_max >= 20) & (r_mean < 20))[0]
print(f'\nmax finds, mean misses: {len(rescued):,}  |  mean finds, max misses: {len(lost):,}')
for i in rescued[:5]:
    print(f"  + {questions[i]['question'][:66]}")
    print(f"    {docs[questions[i]['doc_hash']]['title'][:52]} "
          f"({plen[i]:,} words, answer at word {questions[i]['gold'][0][1]:,})")

In [ ]:
# §7 SAVE. The chunk matrix is the expensive artifact and every later step (reranker
# training pairs, reader context) needs it, so it is pushed even though this notebook only reduces it.
np.save(OUT / 'chunk_emb.npy', E)
np.save(OUT / 'page_emb_chunkmean.npy', P)
json.dump({'doc_hashes': H, 'owner': owner.tolist(), 'model': MODEL,
           'chunk': CHUNK, 'overlap': OVERLAP},
          open(OUT / 'chunk_index.json', 'w'))
print(f"-> {OUT / 'chunk_emb.npy'} ({os.path.getsize(OUT / 'chunk_emb.npy') / 1e9:.2f} GB)")

nqrun.hf_push(str(OUT / 'chunk_emb.npy'), f'{HF_DIR}/chunk_emb.npy',
              f'{MODEL} chunk embeddings, {len(texts):,} chunks @ {CHUNK}/{OVERLAP}')
nqrun.hf_push(str(OUT / 'chunk_index.json'), f'{HF_DIR}/chunk_index.json', 'chunk -> page map')
nqrun.log_metrics('3_chunk_embed', model=MODEL, chunk=CHUNK, overlap=OVERLAP,
                  n_chunks=len(texts), n_pages=len(H), n_questions=len(questions),
                  **{f'{n}_{k}'.replace(' ', '_'): v for n, r in res.items()
                     for k, v in [('r@1', r['recall'][1]), ('r@5', r['recall'][5]),
                                  ('r@10', r['recall'][10]), ('r@20', r['recall'][20]),
                                  ('mrr', r['mrr'])]})